In [0]:
#!pip install lightgbm shap

Config + outputs folder

In [0]:

import os

CITY = "Toronto" 
TABLE = "workspace.capstone_project.toronto_model_ready"
LABEL = "delay_indicator"

cwd = os.getcwd()
print("Current working directory:", cwd)

marker = "DAMO_699-4-Capstone-Project"
if marker not in cwd:
    raise ValueError(
        f"Not inside project folder. Open notebook from Repos/{marker}. cwd={cwd}"
    )

project_root = cwd[:cwd.index(marker) + len(marker)]
OUT_DIR = os.path.join(project_root, "shap_outputs", "shap", CITY.lower())
os.makedirs(OUT_DIR, exist_ok=True)

print("Saving outputs to:", OUT_DIR)

Load data + label sanity check

In [0]:
from pyspark.sql.functions import col

df = (
    spark.table(TABLE)
    .filter(col(LABEL).isNotNull())
    .withColumn(LABEL, col(LABEL).cast("int"))
)

print(f"""
Step 1 — Data sanity ({CITY})

Story:
Before we explain a model, we confirm the target has both outcomes (0 and 1).
If we only have one class, there is no meaningful signal to learn or explain.
""")

df.groupBy(LABEL).count().orderBy(LABEL).show()

labels = [r[LABEL] for r in df.select(LABEL).distinct().collect()]
if len(labels) < 2:
    raise ValueError(f"{CITY}: only one class found in {LABEL}: {labels}")

print(f"{CITY} label check passed — both classes exist.")

Train LightGBM model (SHAP-friendly surrogate)

In [0]:
import numpy as np
import pandas as pd
import lightgbm as lgb

SEED = 42

CATEGORICAL = ["incident_category", "season", "unified_call_source", "location_area"]
NUMERIC     = ["hour", "day_of_week", "month", "year", "unified_alarm_level",
               "calls_past_30min", "calls_past_60min"]

existing = set(df.columns)
cat_cols = [c for c in CATEGORICAL if c in existing]
num_cols = [c for c in NUMERIC if c in existing]

df_model = df.select(*(num_cols + cat_cols + [LABEL]))

print(f"""
Step 2 — Train SHAP-friendly model ({CITY})

Story:
We train a model using real feature names (not hashed),
so that SHAP explanations are readable for stakeholders.
""")
print("Numeric:", num_cols)
print("Categorical:", cat_cols)

train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=SEED)
train_df = train_df.sample(False, 0.35, seed=SEED).limit(200_000)
test_df  = test_df.limit(8_000)

train_pdf = train_df.toPandas()
test_pdf  = test_df.toPandas()

for c in cat_cols:
    train_pdf[c] = train_pdf[c].astype("category")
    test_pdf[c]  = test_pdf[c].astype("category")

X_train = train_pdf[num_cols + cat_cols]
y_train = train_pdf[LABEL].astype(int)

X_exp = test_pdf[num_cols + cat_cols]
y_exp = test_pdf[LABEL].astype(int)

model = lgb.LGBMClassifier(
    n_estimators=700,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=SEED
)
model.fit(X_train, y_train)

print(f"""
Model trained ({CITY})

Story:
Now we compute SHAP values to explain which features push delay risk up/down.
""")

Compute SHAP (FIXED for categorical splits) + validate shapes

In [0]:
import shap

print(f"""
Step 3 — Compute SHAP values ({CITY})

Story:
Because LightGBM uses categorical splits, we must use:
feature_perturbation="tree_path_dependent"
(and no background data).
""")

explainer = shap.TreeExplainer(model, feature_perturbation="tree_path_dependent")

shap_values = explainer.shap_values(X_exp)
shap_matrix = shap_values[1] if isinstance(shap_values, list) else shap_values

print("SHAP shape:", shap_matrix.shape)
print("X shape   :", X_exp.shape)

if shap_matrix.shape != X_exp.shape:
    raise ValueError(f"Shape mismatch. SHAP={shap_matrix.shape}, X={X_exp.shape}")

toronto_shap_values = shap_matrix

print(f"""
SHAP computed + validated ({CITY})

Story:
We can now tell a clear story of what increases delay risk and what reduces it.
""")

Feature importance ranking + CSV export (JIRA schema)

In [0]:
print(f"""
 Step 4 — Feature importance ranking ({CITY})

Story:
We summarize many SHAP explanations into one global ranking using mean(|SHAP|).
This gives the “top drivers” list required by JIRA.
""")

mean_abs = np.mean(np.abs(shap_matrix), axis=0)
imp = pd.DataFrame({"feature": X_exp.columns, "mean_abs_shap": mean_abs})
imp = imp.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
imp["rank"] = np.arange(1, len(imp) + 1)

csv_path = os.path.join(OUT_DIR, "toronto_shap_importance.csv")
imp[["feature", "mean_abs_shap", "rank"]].to_csv(csv_path, index=False)

print("Saved CSV:", csv_path)
display(spark.createDataFrame(imp.head(12)))

 Professional plot styling + reusable plot helpers (JIRA requirement)

In [0]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 320,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11
})

def save_fig(path):
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    plt.close()

def plot_shap_summary_pretty(shap_matrix, X, title, max_display=15):
    plt.figure(figsize=(12, 7))
    shap.summary_plot(shap_matrix, X, show=False, max_display=max_display)
    plt.title(title)

def plot_shap_bar_pretty(shap_matrix, X, title, max_display=15):
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_matrix, X, plot_type="bar", show=False, max_display=max_display)
    plt.title(title)

def plot_shap_dependence_pretty(shap_matrix, X, feature, title):
    plt.figure(figsize=(11, 6))
    shap.dependence_plot(feature, shap_matrix, X, show=False)
    plt.title(title)

print("""
Plot functions ready.

Story:
We’ll use these reusable functions to generate consistent, professional plots
for reporting + dashboards.
""")

Global plots (summary + top-N bar) + storytelling

In [0]:
TOP_N = 15

print(f"""
Step 6 — Global plots ({CITY})

Story:
1) Beeswarm summary: direction + spread of influence.
2) Bar plot (Top {TOP_N}): clean executive view.
""")

summary_path = os.path.join(OUT_DIR, f"toronto_shap_summary_pretty_top{TOP_N}.png")
bar_path     = os.path.join(OUT_DIR, f"toronto_shap_bar_pretty_top{TOP_N}.png")

plot_shap_summary_pretty(shap_matrix, X_exp, f"Toronto — SHAP Summary (Top {TOP_N} drivers)", max_display=TOP_N)
save_fig(summary_path)

print("""
Story:
Each dot is an incident.
- Right → increases delay risk
- Left → decreases delay risk
Color shows low→high feature values.
""")

plot_shap_bar_pretty(shap_matrix, X_exp, f"Toronto — Global Feature Importance (mean |SHAP|, Top {TOP_N})", max_display=TOP_N)
save_fig(bar_path)

top3 = imp["feature"].head(3).tolist()
print(f"""
Toronto Story headline:
The model’s strongest drivers of delay risk are:
1) {top3[0]}
2) {top3[1]}
3) {top3[2]}
""")

print(" Saved:", summary_path)
print(" Saved:", bar_path)

Dependence plots (top 2 drivers) + save + story

In [0]:
# ============================================
# Step 7 — Professional Dependence Plots
# ============================================

top2 = imp["feature"].head(2).tolist()

print(f"""
Step 7 — Dependence plots ({CITY})

Story:
We zoom into the strongest drivers of delay risk.
We examine:
• Does risk increase steadily?
• Is there a tipping point?
• Are there nonlinear patterns?

Top drivers identified:
{top2}
""")

for f in top2:

    print(f"\n📊 Generating dependence plot for: {f}")

    # Clean categorical handling for high-cardinality features
    if f == "location_area":

        print("Detected high-cardinality feature → using Top 10 aggregation.")

        import numpy as np
        import pandas as pd
        import seaborn as sns
        import matplotlib.pyplot as plt

        shap_df = pd.DataFrame({
            f: X_exp[f],
            "shap": shap_matrix[:, list(X_exp.columns).index(f)]
        })

        # Top 10 by mean |SHAP|
        area_importance = (
            shap_df.groupby(f)["shap"]
            .apply(lambda x: np.mean(np.abs(x)))
            .sort_values(ascending=False)
        )

        top_areas = area_importance.head(10).index.tolist()
        shap_top = shap_df[shap_df[f].isin(top_areas)]

        plt.figure(figsize=(14,6))
        sns.boxplot(data=shap_top, x=f, y="shap", palette="viridis")

        plt.xticks(rotation=45, ha="right")
        plt.title(f"{CITY} — SHAP Dependence (Top 10 {f})")
        plt.ylabel("SHAP Value")
        plt.xlabel(f)
        plt.tight_layout()

    else:
        # Standard pretty dependence for numeric features
        plot_shap_dependence_pretty(
            shap_matrix,
            X_exp,
            f,
            f"{CITY} — SHAP Dependence: {f}"
        )

    # Save dynamically per city
    dep_path = os.path.join(OUT_DIR, f"{CITY.lower()}_dependence_{f}.png")
    save_fig(dep_path)

    print(f"""
Story for {f}:

This plot shows how {f} influences predicted delay risk.

• Positive SHAP → increases delay probability
• Negative SHAP → reduces delay probability

Look for:
- Monotonic increase (steady risk growth)
- Threshold effects (sharp slope change)
- Clusters indicating nonlinear behavior

Operational Insight:
Understanding this relationship helps identify
where interventions could reduce delay risk.
""")

    print(" Saved:", dep_path)

Waterfall “case study” (most professional storytelling plot)

In [0]:
print(f"""
Step 8 — Waterfall case study ({CITY})

Story:
This is the most presentation-ready explanation.
We pick one high-risk incident and show a “receipt”:
what pushed risk up, what pushed it down, and where the final prediction landed.
""")

probs = model.predict_proba(X_exp)[:, 1]
idx_high = int(np.argmax(probs))
x_row = X_exp.iloc[idx_high:idx_high+1]
p = float(probs[idx_high])

sv = explainer.shap_values(x_row)
vals = sv[1][0] if isinstance(sv, list) else sv[0]
base = float(explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value)

exp = shap.Explanation(values=vals, base_values=base, data=x_row.iloc[0].values, feature_names=list(x_row.columns))

plt.figure(figsize=(12, 6))
shap.plots.waterfall(exp, max_display=12, show=False)
plt.title(f"Toronto — Waterfall Explanation (High-risk case, p={p:.3f})")

wf_path = os.path.join(OUT_DIR, "toronto_shap_waterfall_highrisk.png")
save_fig(wf_path)

print(f"""
Toronto Story:
This incident has predicted delay risk ≈ {p:.3f}.
- Right-side bars increased risk (drivers of delay).
- Left-side bars reduced risk (protective factors).
This plot is ideal for stakeholder storytelling.
""")
print(" Saved:", wf_path)

Save SHAP sample parquet + notes.md (JIRA requirement)

In [0]:
print(f"""
Step 9 — Save reusable artifacts ({CITY})

Story:
We save a Git-friendly SHAP sample package and a notes file
so the results can be reused in reporting and dashboards (US5.3, US6.3).
""")

TOPK = 12
SAMPLE_ROWS = min(5000, X_exp.shape[0])

top_feats = imp["feature"].head(TOPK).tolist()
idx = [X_exp.columns.get_loc(f) for f in top_feats]

shap_top = shap_matrix[:SAMPLE_ROWS, :][:, idx]
Xs = X_exp.head(SAMPLE_ROWS)[top_feats].copy()
Ss = pd.DataFrame(shap_top, columns=[f"shap_{f}" for f in top_feats])
out_pdf = pd.concat([Xs.reset_index(drop=True), Ss.reset_index(drop=True)], axis=1)

parquet_path = os.path.join(OUT_DIR, "toronto_shap_values_sample.parquet")
out_pdf.to_parquet(parquet_path, index=False)

notes_path = os.path.join(OUT_DIR, "toronto_shap_notes.md")
top5 = imp.head(5)

lines = []
lines.append("# Toronto — SHAP Interpretation Notes\n")
lines.append("## Top drivers\n")
for _, r in top5.iterrows():
    lines.append(f"- **{r['feature']}** (mean|SHAP|={r['mean_abs_shap']:.6f})")
lines.append("\n## Key patterns\n")
lines.append("- Beeswarm: direction + spread of feature impact.")
lines.append("- Bar chart: global importance ranking.")
lines.append("- Waterfall: one incident explanation (best for storytelling).")
lines.append("- Positive SHAP → higher delay risk; negative SHAP → lower delay risk.")

with open(notes_path, "w") as f:
    f.write("\n".join(lines))

print("Saved:", parquet_path)
print("Saved:", notes_path)

print(f"""
Toronto Wrap-up Story:
We translated model predictions into explanations.
Now Toronto has professional plots + ranking + reusable SHAP artifacts saved in outputs/.
""")